# B2 — Drawdown-Penalised PPO (Reward Shaping Ablation)

**Purpose:** Train three PPO agents with `CjMmDrawdownPenalty` across an α sweep
[0.1, 1.0, 10.0]. Evaluate each on the test set to construct a Pareto frontier of
PnL vs drawdown. Select the best α for B3 comparison.

**Design:**
- Reward: `r_t = ΔPnL − φ·q²·Δt − α·drawdown_t`  (φ=0.01, α swept)
- Same training environment as B1 (6-day concatenated replay).
- VecNormalize applied per-α (each model gets its own normalisation stats).

**Outputs (per α):**
- `models/ppo_b2_alpha{α}_doge.zip`
- `models/vecnorm_b2_alpha{α}.pkl`
- `results/b2_alpha{α}_test_results.csv`

**Also saves:** `results/b2_best_alpha.txt` (best α for B3 warm-start)

In [ ]:
import sys
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = next(
    (p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
     if (p / "procs").exists()),
    pathlib.Path.cwd(),
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from stable_baselines3 import PPO

from procs.gym.experiment_config import ReplayExperimentConfig
from procs.gym.data_loader import load_multi_day
from procs.gym.sb3_wrapper import StableBaselinesTradingEnvironment
from procs.gym.notebook_support import (
    build_multi_day_replay_env,
    make_vecnorm,
    evaluate_rl_per_day,
)
from procs.rewards import CjMmDrawdownPenalty

cfg = ReplayExperimentConfig()
cfg.ensure_artifact_dirs()
print(f"Repo root : {cfg.repo_root}")

## Section 1 — Data Loading and Train/Test Split

In [ ]:
TRAIN_DAYS = 6

daily_S, daily_dt, dates = load_multi_day(str(cfg.datasets_dir), pair=cfg.pair)

train_S     = daily_S[:TRAIN_DAYS]
train_dt    = daily_dt[:TRAIN_DAYS]
train_dates = dates[:TRAIN_DAYS]

test_S      = daily_S[TRAIN_DAYS:]
test_dt     = daily_dt[TRAIN_DAYS:]
test_dates  = dates[TRAIN_DAYS:]


EXPECTED_TEST_DAYS = 23
if len(train_S) != TRAIN_DAYS or len(test_S) != EXPECTED_TEST_DAYS:
    raise ValueError(
        f"Expected {TRAIN_DAYS} train days and {EXPECTED_TEST_DAYS} test days, "
        f"found {len(train_S)} and {len(test_S)}."
    )
if set(map(str, train_dates)) & set(map(str, test_dates)):
    raise ValueError("Train/test date overlap detected.")

train_snapshots = sum(len(S) for S in train_S)

print(f"Training: {len(train_S)} separate days, {train_snapshots:,} total snapshots")
print(f"Test    : {len(test_S)} days ({test_dates[0]} → {test_dates[-1]})")

## Section 2 — α Sweep Training

Each α value trains an independent PPO model with `CjMmDrawdownPenalty`.

**`TOTAL_TIMESTEPS` guide:**
- Quick local test: `200_000`
- Full Snellius: `max(2 * len(S_train), 2_000_000)`

On Snellius this section runs as a SLURM job array (`--array=0-2`), one job per α.

In [ ]:
ALPHAS = [0.1, 1.0, 10.0]
TOTAL_TIMESTEPS = max(train_snapshots, 1_000_000)
print(f"α sweep     : {ALPHAS}")
print(f"Timesteps   : {TOTAL_TIMESTEPS:,}  ({TOTAL_TIMESTEPS/train_snapshots:.2f}× data)")
print(f"PPO kwargs  : {cfg.ppo_kwargs()}")

In [ ]:
for alpha in ALPHAS:
    print(f"\n{'='*60}")
    print(f"Training B2 with α = {alpha}")
    print(f"{'='*60}")

    reward_fn = CjMmDrawdownPenalty(
        per_step_inventory_aversion=cfg.phi,
        drawdown_penalty=alpha,
    )

    env  = build_multi_day_replay_env(train_S, train_dt, cfg, reward_fn=reward_fn, mode="sequential")
    sb3  = StableBaselinesTradingEnvironment(env)
    vn   = make_vecnorm(sb3, cfg, training=True, norm_reward=True)

    model = PPO(
        "MlpPolicy",
        vn,
        **cfg.ppo_kwargs(),
        tensorboard_log=str(cfg.repo_root / "tb_logs" / f"b2_alpha{alpha}"),
        verbose=1,
        device="cpu",
    )
    model.learn(total_timesteps=TOTAL_TIMESTEPS)

    model_stem = f"ppo_b2_alpha{alpha}_doge"
    vn_stem    = f"vecnorm_b2_alpha{alpha}"
    model.save(str(cfg.model_path(model_stem)))
    vn.save(str(cfg.vecnorm_path(vn_stem)))
    print(f"  Saved: {model_stem}.zip, {vn_stem}.pkl")

## Section 3 — Test Evaluation (Days 7–29) for Each α

In [ ]:
results_b2 = {}  # alpha -> DataFrame

for alpha in ALPHAS:
    print(f"Evaluating B2 α={alpha} on {len(test_S)} test days ...")
    model = PPO.load(str(cfg.model_path(f"ppo_b2_alpha{alpha}_doge")), device="cpu")

    df = evaluate_rl_per_day(
        model=model,
        vecnorm_path=cfg.vecnorm_path(f"vecnorm_b2_alpha{alpha}"),
        test_S=test_S,
        test_dt=test_dt,
        test_dates=test_dates,
        config=cfg,
        seed=42,
    )
    results_b2[alpha] = df

    out_path = cfg.result_path(f"b2_alpha{alpha}_test_results.csv")
    df.to_csv(out_path)
    print(f"  Sharpe mean={df['Sharpe'].mean():.3f}  MaxDD mean={df['Max DD'].mean():.4f}")
    print(f"  Saved → {out_path}")

## Section 4 — Pareto Frontier Plot

In [ ]:
# Also load B1 for reference
try:
    df_b1 = pd.read_csv(cfg.result_path("b1_test_results.csv"), index_col="Day")
    b1_available = True
except FileNotFoundError:
    print("B1 results not found — run nb1 first to include B1 on the Pareto plot.")
    b1_available = False

fig, ax = plt.subplots(figsize=(9, 6))

colors = ["steelblue", "darkorange", "seagreen"]
for (alpha, df), color in zip(results_b2.items(), colors):
    ax.scatter(
        df["Max DD"].mean(),
        df["Final PnL"].mean(),
        s=150, color=color, zorder=5,
        label=f"B2 α={alpha}  (MaxDD={df['Max DD'].mean():.4f}, PnL={df['Final PnL'].mean():.2f})",
    )

if b1_available:
    ax.scatter(
        df_b1["Max DD"].mean(),
        df_b1["Final PnL"].mean(),
        s=150, color="crimson", marker="^", zorder=5,
        label=f"B1 Unconstrained  (MaxDD={df_b1['Max DD'].mean():.4f}, PnL={df_b1['Final PnL'].mean():.2f})",
    )

ax.set_xlabel("Mean Max Drawdown (lower is better →)")
ax.set_ylabel("Mean Final PnL (higher is better ↑)")
ax.set_title("B2 Pareto Frontier: PnL vs Max Drawdown\n(each point = 23 test-day mean)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(cfg.result_path("b2_pareto_frontier.png")), dpi=150, bbox_inches="tight")
plt.show()

## Section 5 — Select Best α for B3 Comparison

Best α = smallest MaxDD among models with positive mean Sharpe.
This becomes the primary B2 comparison point against B3 in nb3 and nb4.

In [ ]:
alpha_metrics = []
for alpha, df in results_b2.items():
    alpha_metrics.append({
        "alpha":      alpha,
        "mean_sharpe": df["Sharpe"].mean(),
        "mean_maxdd":  df["Max DD"].mean(),
        "mean_pnl":    df["Final PnL"].mean(),
    })

df_alpha = pd.DataFrame(alpha_metrics).set_index("alpha")
print("=== α Sweep Summary ===")
print(df_alpha.to_string(float_format="{:.4f}".format))

positive_sharpe = df_alpha[df_alpha["mean_sharpe"] > 0]
if len(positive_sharpe) == 0:
    print("WARNING: No α has positive Sharpe — selecting smallest α as fallback.")
    best_alpha = min(ALPHAS)
else:
    best_alpha = float(positive_sharpe["mean_maxdd"].idxmin())

print(f"\nBest α = {best_alpha}  (min MaxDD with Sharpe > 0)")

best_alpha_path = cfg.result_path("b2_best_alpha.txt")
with open(best_alpha_path, "w") as f:
    f.write(str(best_alpha))
print(f"Saved best α → {best_alpha_path}")

df_b2_best = results_b2[best_alpha]
b2_primary_path = cfg.result_path("b2_test_results.csv")
df_b2_best.to_csv(b2_primary_path)
print(f"Saved selected B2 results -> {b2_primary_path}")


## Section 6 — Results Display

In [ ]:
pd.set_option("display.float_format", "{:.4f}".format)
METRICS = ["Sharpe", "Sortino", "Max DD", "P&L-to-MAP", "Final PnL"]

for alpha, df in results_b2.items():
    tag = " ← BEST" if alpha == best_alpha else ""
    print(f"\n=== B2 α={alpha}{tag} — mean ± std ===")
    summary = pd.DataFrame({"Mean": df[METRICS].mean(), "Std": df[METRICS].std()}).T
    print(summary.to_string())